# To perform and visualize analyses online 

## Before the experiment: get everything ready 

### imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import hydra
from hydra.utils import get_original_cwd
import os
from omegaconf import DictConfig, OmegaConf
from dataclasses import dataclass
from typing import List, Dict, Any

from IPython.display import display



In [3]:
# Load config
import sys
import os
from pathlib import Path


# Add the parent directory to the path so we can import modules properly
cwd = Path.cwd()
print(f"home directory: {cwd}")
relative_repo_path = "GitRepos/simulation_closed_loop"

# append repo path 
sys.path.append(str(cwd / relative_repo_path))

# Import Hydra config utilities
from omegaconf import DictConfig, OmegaConf
import hydra
from hydra.utils import instantiate
from hydra.core.config_store import ConfigStore
from hydra import compose, initialize

# Initialize Hydra with the relative path to the config directory
config_path = os.path.join(relative_repo_path,"config")
print(f"Config path: {config_path}")

# Initialize Hydra
with initialize(version_base="1.3", config_path=config_path):
    # Compose the configuration
    cfg = compose(config_name="config")

# Print the config to verify it loaded correctly
print("Configuration loaded successfully:")
print(OmegaConf.to_yaml(cfg))



home directory: /gpfs01/euler/User/ssuhai
Config path: GitRepos/simulation_closed_loop/config
Configuration loaded successfully:
data_subfolders:
  day: 20250717
  experiment: 1
DJ:
  username: ssuhai
  userinfo:
    experimenter: closedlooptest
    animal_loc: 1
    region_loc: 2
    field_loc: 3
    stimulus_loc: 4
    cond1_loc: 5
    data_dir: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/klaudia_data_test
  table_parameters:
    PreprocessParams:
    - preprocess_id: 1
      fs_resample: 60
      stim_names:
      - gChirp
      - lChirp
      - movingbar
      - densenoise
    - preprocess_id: 2
      window_length: 60
      poly_order: 3
      non_negative: 1
      subtract_baseline: 0
      standardize: 1
      stim_names:
      - mouse_cam
    Stimulus:
      noise:
        stim_name: densenoise
        stim_family: noise
        pix_n_x: 20
        pix_n_y: 15
        skip_duplicates: true
        pix_scale_x_um: 40
        pix_scale_y_um: 40
     

In [4]:
from simulations.loop_components.dj_wrappers import DJTableHolder,Preprocessor,QualityAndTypeWrapper,STAWrapper,RandomSeedMEIWrapper
from simulations.loop_components.recording_file_copier import copy_rec_files,create_directory_structure
from simulations.loop_components.stimulus import create_single_mei_avis_and_metadata
from simulations.loop_components.utils import log

### Create processing components (connect them to DB)

In [5]:
# create preprocessor
os.environ["DJ_SUPPORT_FILEPATH_MANAGEMENT"] = "TRUE"

dj_table_holder = DJTableHolder(
                username=cfg.DJ.username, # type: ignore
                
                #paths
                home_directory=cfg.paths.home_directory, # type: ignore
                repo_directory=cfg.paths.repo_directory, # type: ignore
                dj_config_directory= cfg.paths.dj_config_directory, # type: ignore
                rgc_output_directory= cfg.paths.rgc_output_directory, # type: ignore
                data_subfolders=cfg.data_subfolders, # type: ignore


                userinfo= cfg.DJ.userinfo, # type: ignore

                table_parameters=cfg.DJ.table_parameters, # type: ignore

                # from overall configs
                debug=cfg.debug, # type: ignore
                plot_results=cfg.plot_results, # type: ignore
                )



In [6]:
# # # Load config and tables
# dj_table_holder.load_config()
# dj_table_holder.load_tables()
# # # print(" loaded and configured successfully")
# # # # dj_table_holder.clear_tables("experiment")

dj_table_holder.setup()



[2025-09-02 13:32:04,078][INFO]: Connecting ssuhai@172.25.240.200:3306
[2025-09-02 13:32:04,133][INFO]: Connected ssuhai@172.25.240.200:3306


schema_name: ageuler_ssuhai_closed_loop
Done reconnecting. Skipping adding new entries from config.


/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/simulations/loop_components/dj_wrappers.py:301: UserWarning: 
Some DJ tables (like UserInfo) are not empty, skipping adding new entries from config.
Make sure this is wanted. Call clear_tables(`all`) if you want different data in there
  warnings.warn("\nSome DJ tables (like UserInfo) are not empty, skipping adding new entries from config.\nMake sure this is wanted. Call clear_tables(`all`) if you want different data in there")


In [7]:
preprocessor = Preprocessor(dj_table_holder=dj_table_holder)


quality_type_analysis_wrapper = QualityAndTypeWrapper(
    dj_table_holder=dj_table_holder,)

sta_wrapper = STAWrapper(
    dj_table_holder=dj_table_holder,)

random_seed_mei_wrapper = RandomSeedMEIWrapper(
    dj_table_holder=dj_table_holder,
    model_configs=cfg.model_configs, # type: ignore
    mei_generation_params=cfg.openretina.mei_generation, # type: ignore
    seeds=[111,222]
)

## During the experimet

### Move files from server to the repo 

In [12]:
create_directory_structure(base_directory= cfg.DJ.userinfo.data_dir,
                           date=  cfg.data_subfolders.day, 
                           experiment = cfg.data_subfolders.experiment)

copy_rec_files(
    recording_files_dir=cfg.paths.recording_files_dir,  # type: ignore
    destination_base=cfg.DJ.userinfo.data_dir,  # type: ignore
    date=cfg.data_subfolders.day,  # type: ignore
    experiment=cfg.data_subfolders.experiment,  # type: ignore
    full_dummy_ini_dir= os.path.join(cfg.paths.repo_directory, cfg.paths.dummy_ini_dir),  # type: ignore
)

SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/klaudia_data_test/20250717/1/Raw/M1_LR_GCL0_Chirp_iter0.smh already exists, skipping.
SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/klaudia_data_test/20250717/1/Raw/M1_LR_GCL0_MB_iter0.smh already exists, skipping.
SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/klaudia_data_test/20250717/1/Raw/M1_LR_GCL0_MC18_iter0.smh already exists, skipping.
SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/klaudia_data_test/20250717/1/Raw/M1_LR_GCL0_Chirp_iter0.smp already exists, skipping.
SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/klaudia_data_test/20250717/1/Raw/M1_LR_GCL0_MC18_iter0.smp already exists, skipping.
SKIPPING File /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/klaudia_data_test/20250717/1/Raw/M1_LR_GCL0_DN_

In [15]:
preprocessor.upload_iteration_metadata()

Scanning for experimenter: closedlooptest
	header_path: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/klaudia_data_test/20250717/1
		header_name: 20250717_left.ini
		Adding: {'experimenter': 'closedlooptest', 'date': datetime.datetime(2025, 7, 17, 0, 0), 'exp_num': 1}


OpticDisk: 100%|██████████| 1/1 [00:00<00:00, 31.41it/s]


Found 4 files in 1 fields for key={'experimenter': 'closedlooptest', 'date': datetime.date(2025, 7, 17), 'exp_num': 1, 'raw_id': 1}
	Adding field: `{'field': 'GCL0', 'region': 'LR', 'cond1': 'iter0', 'experimenter': 'closedlooptest', 'date': datetime.date(2025, 7, 17), 'exp_num': 1, 'raw_id': 1}`


Processes: 100%|██████████| 6/6 [00:08<00:00,  1.39s/it]


In [9]:
missing_keys = dj_table_holder("RoiMask")().list_missing_field()
if len(missing_keys) == 1:
    field_key = missing_keys[0]
elif len(missing_keys) > 1:
    raise ValueError(f"Multiple missing fields found: {missing_keys}")
else:
    print("No missing fields found, using the last field key.")
    all_field_key = dj_table_holder("Field")().proj().fetch(as_dict=True)
    field_key = all_field_key[-1]
    print(f"Field key: {field_key}")

No missing fields found, using the last field key.
Field key: {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 7, 17), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter0'}


In [10]:
quality_wrapper_filtering_params = {
    "d_qi_min":0.5,
    "qidx_min": 0.5,
    "celltypes": list(range(1,33)),
    "classifier_confidence": 0.25
}
sta_wrapper_filtering_parameters = {
    "rf_qidx_min": 0.5,}



In [16]:
dj_table_holder("RoiMask")()

experimenter name of the experimenter,date date of recording,exp_num experiment number in a day,raw_id unique param set id,field string identifying files corresponding to field,region region (e.g. LR or RR),cond1 condition (pharmacological or other),stim_name Unique string identifier,cond2 condition (pharmacological or other),roi_mask ROI mask for recording field
closedlooptest,2025-07-17,1,1,GCL0,LR,iter0,gChirp,control,=BLOB=


In [ ]:
#dj_table_holder.clear_tables("rois",field_key=field_key)

[2025-08-21 20:12:37,706][INFO]: Deleting 4 rows from `ageuler_ssuhai_closed_loop`.`roi_mask__roi_mask_presentation`
[2025-08-21 20:12:37,731][INFO]: Deleting 1 rows from `ageuler_ssuhai_closed_loop`.`roi_mask`
[2025-08-21 20:12:39,828][INFO]: Deletes committed.


Removed file: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/klaudia_data_test/20250717/1/ROIs/M1_LR_GCL0_DN_iter0_ROIs.pkl
Removed file: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/klaudia_data_test/20250717/1/ROIs/M1_LR_GCL0_Chirp_iter0_ROIs.pkl
Removed file: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/klaudia_data_test/20250717/1/ROIs/M1_LR_GCL0_MC18_iter0_ROIs.pkl
Removed file: /gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/data/recordings/klaudia_data_test/20250717/1/ROIs/M1_LR_GCL0_MB_iter0_ROIs.pkl


In [21]:
# compute 
preprocessor.add_iteration_roi_mask(field_key=field_key)
preprocessor.add_iteration_rois()
preprocessor.add_iteration_traces()


field_key: {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 7, 17), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter0'} 
pres_keys: [{'experimenter': 'closedlooptest', 'date': datetime.date(2025, 7, 17), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter0', 'stim_name': 'densenoise', 'cond2': 'control'}, {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 7, 17), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter0', 'stim_name': 'gChirp', 'cond2': 'control'}, {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 7, 17), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter0', 'stim_name': 'mouse_cam', 'cond2': 'control'}, {'experimenter': 'closedlooptest', 'date': datetime.date(2025, 7, 17), 'exp_num': 1, 'raw_id': 1, 'field': 'GCL0', 'region': 'LR', 'cond1': 'iter0', 'stim_name': 'movingbar', 'cond2': 'control'}]
No ROI masks found for field: {'experimenter

Returned InteractiveRoiCanvas object. To start GUI, call <enter_object_name>.start_gui().


Processes: 100%|██████████| 428/428 [00:00<00:00, 547.79it/s]


In [ ]:
quality_type_analysis_wrapper.compute_analysis(
    field_key=field_key)

# filter 
passed_roi_ids_chirp_mb = quality_type_analysis_wrapper.get_roi_ids_passing_criterion(field_key=field_key,
    d_qi_min =cfg.quality_filtering["d_qi_min"],
    qidx_min=cfg.quality_filtering["chirp_qi_min"],
    celltypes=cfg.quality_filtering["celltypes"],
    classifier_confidence=quality_wrapper_filtering_params["classifier_confidence"])
if len(passed_roi_ids_chirp_mb) == 0:
    raise ValueError("No ROIs passed the quality criterion for quality and type.")
print(f"ROIs passed quality chirp mb filtering: {passed_roi_ids_chirp_mb}")



ROIs passed quality chirp mb filtering: [12, 15, 18, 27, 30, 36, 42, 45, 63, 66, 69, 78, 81, 90, 93, 102]


In [25]:
sta_wrapper.clear_tables(field_key=field_key)

[2025-09-02 11:00:41,806][INFO]: Deleting 16 rows from `ageuler_ssuhai_closed_loop`.`__s_t_a__data_set`
[2025-09-02 11:00:41,882][INFO]: Deleting 16 rows from `ageuler_ssuhai_closed_loop`.`__fit_gauss2_d_r_f`
[2025-09-02 11:00:41,911][INFO]: Deleting 16 rows from `ageuler_ssuhai_closed_loop`.`__split_r_f`
[2025-09-02 11:00:41,939][INFO]: Deleting 16 rows from `ageuler_ssuhai_closed_loop`.`__s_t_a`
[2025-09-02 11:00:41,968][INFO]: Deleting 16 rows from `ageuler_ssuhai_closed_loop`.`__d_noise_trace`
[2025-09-02 11:00:44,284][INFO]: Deletes committed.
[2025-09-02 11:00:44,316][INFO]: Deleting 0 rows from `ageuler_ssuhai_closed_loop`.`__s_t_a`
[2025-09-02 11:00:44,317][WARNING]: Nothing to delete.
[2025-09-02 11:00:44,348][INFO]: Deleting 0 rows from `ageuler_ssuhai_closed_loop`.`__split_r_f`
[2025-09-02 11:00:44,349][WARNING]: Nothing to delete.
[2025-09-02 11:00:44,379][INFO]: Deleting 0 rows from `ageuler_ssuhai_closed_loop`.`__fit_gauss2_d_r_f`
[2025-09-02 11:00:44,380][WARNING]: Nothi

In [26]:
# sta 
sta_wrapper.compute_analysis(
    field_key=field_key,
    roi_id_subset=passed_roi_ids_chirp_mb,)


Processes: 100%|██████████| 16/16 [00:00<00:00, 17.71it/s]


In [20]:
assert (dj_table_holder("STA")().fetch("roi_id") == passed_roi_ids_chirp_mb).all(), "STA roi_id does not match passed roi_ids from quality and type filtering."

In [ ]:
# filter
passed_roi_ids_sta = sta_wrapper.get_roi_ids_passing_criterion(field_key=field_key,
                                                               rf_qidx_min=cfg.quality_filtering["rf_qidx_min"])
if len(passed_roi_ids_sta) == 0:
    raise ValueError("No ROIs passed the quality criterion for STA.")
print(f"ROIs passed STA filtering: {passed_roi_ids_sta}")


ROIs passed STA filtering: [15, 27, 30, 36, 42, 45, 63, 66, 69, 78, 81, 90, 93]


In [12]:
dj_table_holder.multiprocessing_threads = 1

In [25]:
random_seed_mei_wrapper.mei_data_container.columns

Index(['readout_idx', 'roi_id', 'mei_id', 'seed', 'mei', 'temporal_kernels',
       'spatial_kernels', 'stability', 'responses_all_readout_idx',
       'mean_responses_all_readout_idx'],
      dtype='object')

In [42]:
random_seed_mei_wrapper.plot1(roi_id = 78)

ROI 78 does not have an MEI. Select among the following: 
[15, 27, 30, 42, 45, 63, 66, 69, 81, 90]


In [ ]:
random_seed_mei_wrapper.compute_analysis(field_key=field_key,
                                         roi_id_subset=passed_roi_ids_sta,)



33
Original dataset contains 13 neurons over 1 fields
 ------------------------------------ 
Dropped 0 fields that did not contain the target cell types (1 remaining)
Overall, dropped 0 neurons of non-target cell types (-0.00%).
 ------------------------------------ 
Dropped 0 fields with quality indices below threshold (1 remaining)
Overall, dropped 1 neurons over quality checks (-7.69%).
 ------------------------------------ 
Dropped 0 fields with classifier confidences below 0.25
Overall, dropped 0 neurons with classifier confidences below 0.25 (-0.00%).
 ------------------------------------ 
 ------------------------------------ 
Final dataset contains 12 neurons over 1 fields
Total number of cells dropped: 1 (-7.69%)


Upsampling natural spikes traces to get final responses.:   0%|          | 0/1 [00:00<?, ?it/s]

Creating movie dataloaders:   0%|          | 0/1 [00:00<?, ?it/s]

Seed set to 2000
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA A100-PCIE-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
2025-09-02 13:34:42.230660: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/gpfs01/euler/User/ssuhai/.local/lib/python3.12/site-packages/lightning/fabric/loggers/csv_logs.py:268: Experiment logs directory output/csv/ exists and is not empty. Previous log files in this directory will be deleted when the new ones are saved!
/gpfs01/euler/User/ssuhai/.local/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint direct

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

Generating unstable MEIs for neurons (readout idx): [2, 6].


Seed set to 222


Done with meis in phase unstable.
Start decomposing ...
Decomposing MEIs for neuron (readout idx) 2 ...
Decomposing MEIs for neuron (readout idx) 6 ...
Generating stable MEIs for neurons (readout idx): [0, 1, 4, 5, 7, 8, 9, 10].


Seed set to 111


Done with meis in phase stable.
Start decomposing ...
Decomposing MEIs for neuron (readout idx) 0 ...
Decomposing MEIs for neuron (readout idx) 1 ...
Decomposing MEIs for neuron (readout idx) 4 ...
Decomposing MEIs for neuron (readout idx) 5 ...
Decomposing MEIs for neuron (readout idx) 7 ...
Decomposing MEIs for neuron (readout idx) 8 ...
Decomposing MEIs for neuron (readout idx) 9 ...
Decomposing MEIs for neuron (readout idx) 10 ...
Generating responses for neurons in readout 10 to all meis 12 ...


In [38]:
print(2)
random_seed_mei_wrapper.mei_subanalysis()

[2025-09-02 12:59:08,733][WARNING]: MySQL server has gone away. Reconnecting to the server.
--- Logging error ---
Traceback (most recent call last):
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/datajoint/connection.py", line 343, in query
    self._execute_query(cursor, query, args, suppress_warnings)
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/datajoint/connection.py", line 299, in _execute_query
    raise translate_query_error(err, query)
datajoint.errors.LostConnectionError: ('Server connection lost', 'Lost connection to MySQL server during query')

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/logging/__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
ValueError: I/O operation on closed file.
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 8

2


Seed set to 111


Generating unstable MEIs for neurons (readout idx): [0, 4].


Seed set to 222


Start decomposing ...
Generating stable MEIs for neurons (readout idx): [1, 2, 5, 6, 7, 8].


Seed set to 111


Start decomposing ...


KeyboardInterrupt: 

In [ ]:
random_seed_mei_wrapper.get_responses_and_add_to_container()


In [56]:
random_seed_mei_wrapper.mei_data_container[["readout_idx","stability","mean_responses_all_readout_idx"]]

,readout_idx,stability,mean_responses_all_readout_idx
0,0,unstable,"[3.0669007, 1.0788754, 1.297539, 1.2112863, 2...."
1,0,unstable,"[2.6983378, 0.85414314, 1.2265472, 1.2419125, ..."
2,4,unstable,"[2.7701602, 0.8902386, 1.1621171, 1.1370437, 2..."
3,4,unstable,"[2.5725436, 0.77136815, 1.0902089, 1.0614189, ..."
4,1,stable,"[2.805798, 5.4680963, 4.30828, 2.7747808, 2.34..."
5,2,stable,"[1.7774045, 5.565527, 5.451215, 2.815146, 1.49..."
6,5,stable,"[2.4266973, 5.3656087, 4.629107, 2.9815748, 1...."
7,6,stable,"[2.2289875, 6.3084893, 6.1097903, 2.7270772, 2..."
8,7,stable,"[1.8777401, 5.3634477, 4.9427295, 2.8802845, 1..."
9,8,stable,"[2.7347972, 5.199195, 4.1092043, 2.9845207, 2...."


In [1]:
print("eef")

eef


In [ ]:
#random_seed_mei_wrapper.upload_to_db()

In [49]:
dj_table_holder("OnlineMEIs")()

[2025-08-24 17:06:11,581][WARNING]: MySQL server has gone away. Reconnecting to the server.
--- Logging error ---
Traceback (most recent call last):
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/datajoint/connection.py", line 343, in query
    self._execute_query(cursor, query, args, suppress_warnings)
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages/datajoint/connection.py", line 299, in _execute_query
    raise translate_query_error(err, query)
datajoint.errors.LostConnectionError: ('Connection timed out', "MySQL server has gone away (SSLEOFError(8, 'EOF occurred in violation of protocol (_ssl.c:2417)'))")

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/.pyenv/versions/miniconda3-latest/lib/python3.12/logging/__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
ValueError: I/O operation on closed file.
Call stack:
  File "<frozen runpy>", line 198, in

experimenter name of the experimenter,date date of recording,exp_num experiment number in a day,raw_id unique param set id,field string identifying files corresponding to field,region region (e.g. LR or RR),cond1 condition (pharmacological or other),"session_name session name,",roi_id,readout_idx the index of the readout neuron in the model,seed the seed used to generate the MEI,"mei torch.tensor of shape (batch,channel,time,height,width)"
closedlooptest,2025-07-17,1,1,GCL0,LR,iter0,online_session_1_ventral1_20250717,21,1,111,=BLOB=
closedlooptest,2025-07-17,1,1,GCL0,LR,iter0,online_session_1_ventral1_20250717,21,1,222,=BLOB=
closedlooptest,2025-07-17,1,1,GCL0,LR,iter0,online_session_1_ventral1_20250717,24,2,111,=BLOB=
closedlooptest,2025-07-17,1,1,GCL0,LR,iter0,online_session_1_ventral1_20250717,24,2,222,=BLOB=
closedlooptest,2025-07-17,1,1,GCL0,LR,iter0,online_session_1_ventral1_20250717,27,3,111,=BLOB=
closedlooptest,2025-07-17,1,1,GCL0,LR,iter0,online_session_1_ventral1_20250717,27,3,222,=BLOB=
closedlooptest,2025-07-17,1,1,GCL0,LR,iter0,online_session_1_ventral1_20250717,36,4,111,=BLOB=
closedlooptest,2025-07-17,1,1,GCL0,LR,iter0,online_session_1_ventral1_20250717,36,4,222,=BLOB=
closedlooptest,2025-07-17,1,1,GCL0,LR,iter0,online_session_1_ventral1_20250717,39,5,111,=BLOB=
closedlooptest,2025-07-17,1,1,GCL0,LR,iter0,online_session_1_ventral1_20250717,39,5,222,=BLOB=


In [9]:
# somehow I get a circular import error if I import this at the top
from simulations.gui.integrated_autorois import InteractiveRoiCanvas

In [10]:
online_analysis_gui = InteractiveRoiCanvas(
    dj_table_holder=dj_table_holder,
    dj_preprocessor=preprocessor,
    all_dj_wrappers=[quality_type_analysis_wrapper,sta_wrapper,random_seed_mei_wrapper],
    field_key=field_key,
    take_roi_rgba_from_this_analysis=random_seed_mei_wrapper.name,  # specify which analysis to take the roi color and alpha from
    canvas_width=30,
    )

NameError: name 'field_key' is not defined

In [79]:
display(online_analysis_gui.start_gui())

In [36]:
display(online_analysis_gui.start_gui())

# The stimulus

In [37]:
roi_ids,seeds =dj_table_holder("OnlineMEIs")().fetch("roi_id","seed")

In [52]:
roi_seed = list(zip(roi_ids[:7],seeds[:7]))
print(f"roi_seed: {roi_seed}")

roi_seed: [(21, 111), (21, 222), (24, 111), (24, 222), (27, 111), (27, 222), (36, 111)]


In [ ]:
roi_id2mei_id,roi_id2mei_info = random_seed_mei_wrapper.select_subset_of_meis_for_each_roi()

2


In [18]:
random_seed_mei_wrapper.mei_data_container[["readout_idx","stability","roi_id","seed","mei_id"]]

,readout_idx,stability,roi_id,seed,mei_id
0,2,unstable,30,111,roi_30_seed_111
1,2,unstable,30,222,roi_30_seed_222
2,6,unstable,63,111,roi_63_seed_111
3,6,unstable,63,222,roi_63_seed_222
4,0,stable,15,111,roi_15_seed_111
5,1,stable,27,111,roi_27_seed_111
6,4,stable,42,111,roi_42_seed_111
7,5,stable,45,111,roi_45_seed_111
8,7,stable,66,111,roi_66_seed_111
9,8,stable,69,111,roi_69_seed_111


In [19]:
roi_seed = [(30,111),
            (30,222),
            (63,111),
            (63,222),
            (15,111),
            (27,111),
            (42,111),
            (45,111)]

In [22]:
print(2)
create_single_mei_avis_and_metadata(
    rois_seed=roi_seed,
    roi_id2mei_ids = roi_id2mei_id,    
    mei_data_container=random_seed_mei_wrapper.mei_data_container,
    stimulus_table=dj_table_holder("Stimulus")(),
    fit_gauss_2d_rf_table= dj_table_holder("FitGauss2DRF")(),
    abs_save_dir=cfg.paths.stimulus_output_dir,
)

2
Found 6 rois in the FitGauss2D table.
Initial roi_id order: [15, 27, 30, 42, 45, 63].
Reordered roi_id order: [42, 27, 63, 30, 45, 15].
Generated MEI presentation ordering: [['roi_42_seed_111', 'roi_81_seed_111', 'roi_69_seed_111', 'roi_27_seed_111', 'roi_66_seed_111', 'roi_30_seed_111'], ['roi_45_seed_111', 'roi_66_seed_111', 'roi_81_seed_111', 'roi_27_seed_111', 'roi_63_seed_222', 'roi_30_seed_111'], ['roi_27_seed_111', 'roi_63_seed_222', 'roi_42_seed_111', 'roi_69_seed_111', 'roi_63_seed_111', 'roi_90_seed_111'], ['roi_30_seed_111', 'roi_66_seed_111', 'roi_42_seed_111', 'roi_90_seed_111', 'roi_69_seed_111', 'roi_30_seed_222'], ['roi_90_seed_111', 'roi_63_seed_222', 'roi_45_seed_111', 'roi_30_seed_111', 'roi_27_seed_111', 'roi_69_seed_111'], ['roi_42_seed_111', 'roi_30_seed_111', 'roi_81_seed_111', 'roi_69_seed_111', 'roi_66_seed_111', 'roi_15_seed_111']].
Creating MEI avi file for roi_30_seed_111 at /gpfs01/euler/data/Data/Suhai/stimuli_closed_loop/iter9/mei_roi_30_seed_111.avi.
C

# Clean up

In [32]:
userinput = input("Cleanup? (y/n): ")
if userinput.lower() == 'y':
    dj_table_holder.clear_tables("all")


[2025-08-12 18:21:27,897][INFO]: Deleting 3 rows from `ageuler_ssuhai_closed_loop`.`__field__stack_averages`
[2025-08-12 18:21:27,964][INFO]: Deleting 3 rows from `ageuler_ssuhai_closed_loop`.`__presentation__scan_info`
[2025-08-12 18:21:28,010][INFO]: Deleting 9 rows from `ageuler_ssuhai_closed_loop`.`__presentation__stack_averages`
[2025-08-12 18:21:28,150][INFO]: Deleting 97 rows from `ageuler_ssuhai_closed_loop`.`__peak_s_t_a_position`
[2025-08-12 18:21:28,200][INFO]: Deleting 97 rows from `ageuler_ssuhai_closed_loop`.`__s_t_a__data_set`
[2025-08-12 18:21:28,253][INFO]: Deleting 97 rows from `ageuler_ssuhai_closed_loop`.`__split_r_f`
[2025-08-12 18:21:28,290][INFO]: Deleting 97 rows from `ageuler_ssuhai_closed_loop`.`__s_t_a`
[2025-08-12 18:21:28,323][INFO]: Deleting 97 rows from `ageuler_ssuhai_closed_loop`.`__d_noise_trace`
[2025-08-12 18:21:28,465][INFO]: Deleting 95 rows from `ageuler_ssuhai_closed_loop`.`__celltype_assignment`
[2025-08-12 18:21:28,496][INFO]: Deleting 95 rows 